# Stock Filtering Based on Financial Metrics

This notebook filters Korean stocks based on:
- **영업이익률 (Operating Profit Margin)** >= 8%
- **ROE (Return on Equity)** >= 10%
- **부채비율 (Debt Ratio)** <= 150%

## Step 1: Setup and Load Financial Data

In [184]:
import pandas as pd
import numpy as np

# Define filtering thresholds
OPM_MIN = 8.0      # 영업이익률 >= 8%
ROE_MIN = 10.0     # ROE >= 10%
DEBT_MAX = 150.0   # 부채비율 <= 150%

print("Filtering criteria:")
print(f"  영업이익률 >= {OPM_MIN}%")
print(f"  ROE >= {ROE_MIN}%")
print(f"  부채비율 <= {DEBT_MAX}%")

Filtering criteria:
  영업이익률 >= 8.0%
  ROE >= 10.0%
  부채비율 <= 150.0%


In [185]:
# Load financial data
print("Loading 재무데이터.csv...")
df_financial = pd.read_csv('data/재무데이터.csv', encoding='utf-8-sig')

print(f"Shape: {df_financial.shape}")
print(f"\nColumns: {list(df_financial.columns[:5])}...")
df_financial.head()

Loading 재무데이터.csv...
Shape: (113042, 50)

Columns: ['Symbol', 'Symbol Name', 'Item Name ', '2014-03-31', '2014-06-30']...


,Symbol,Symbol Name,Item Name,2014-03-31,2014-06-30,2014-09-30,2014-12-31,2015-03-31,2015-06-30,2015-09-30,...,2023-06-30,2023-09-30,2023-12-31,2024-03-31,2024-06-30,2024-09-30,2024-12-31,2025-03-31,2025-06-30,2025-09-30
0,A005930,삼성전자,매출액(천원),"53,675,326,000","52,353,229,000","47,447,310,000","52,730,122,000","47,117,918,000","48,537,539,000","51,682,572,000",...,"60,005,533,000","67,404,652,000","67,779,938,000","71,915,601,000","74,068,302,000","79,098,731,000","75,788,269,000","79,140,503,000","74,566,317,000","86,061,747,000"
1,A005930,삼성전자,영업이익(천원),"8,488,799,000","7,187,323,000","4,060,522,000","5,288,427,000","5,979,367,000","6,897,937,000","7,393,373,000",...,"668,547,000","2,433,534,000","2,824,717,000","6,606,009,000","10,443,878,000","9,183,371,000","6,492,703,000","6,685,272,000","4,676,057,000","12,166,062,000"
2,A005930,삼성전자,세전계속사업이익(천원),"9,648,970,000","7,785,106,000","4,846,871,000","5,594,087,000","6,218,643,000","7,628,024,000","7,390,119,000",...,"1,712,995,000","3,942,601,000","3,524,289,000","7,706,723,000","11,595,344,000","10,320,412,000","7,907,255,000","9,151,576,000","5,756,129,000","13,545,563,000"
3,A005930,삼성전자,지배주주순이익(천원),"7,484,680,000","6,176,506,000","4,135,422,000","5,285,891,000","4,519,323,000","5,626,734,000","5,306,104,000",...,"1,547,018,000","5,501,304,000","6,023,827,000","6,621,030,000","9,642,653,000","9,781,547,000","7,576,133,000","8,028,407,000","4,934,034,000","12,006,461,000"
4,A005930,삼성전자,비지배주주순이익(천원),"89,761,000","74,275,000","86,923,000","60,900,000","106,492,000","125,563,000","152,455,000",...,"176,553,000","342,867,000","320,931,000","133,678,000","198,692,000","319,357,000","178,261,000","194,471,000","182,401,000","219,286,000"


## Step 2: Extract and Calculate Financial Metrics

In [186]:
# Identify the most recent quarter column (last date column)
# Note: 'Item Name' column has a trailing space in the CSV
date_columns = [col for col in df_financial.columns if col not in ['Symbol', 'Symbol Name', 'Item Name ', 'Item Name']]
latest_quarter = date_columns[-1]

print(f"Using most recent quarter: {latest_quarter}")
print(f"Total date columns available: {len(date_columns)}")

Using most recent quarter: 2025-09-30
Total date columns available: 47


In [187]:
# Check unique Item Names
# Note: Column name is 'Item Name ' with a trailing space
item_col = 'Item Name ' if 'Item Name ' in df_financial.columns else 'Item Name'
print(f"Using column: '{item_col}'")
print("\nAvailable Item Names:")
print(df_financial[item_col].unique())

Using column: 'Item Name '

Available Item Names:
<ArrowStringArray>
[                  '매출액(천원)',                  '영업이익(천원)',
              '세전계속사업이익(천원)',               '지배주주순이익(천원)',
              '비지배주주순이익(천원)',              '순이익 (E3)(억원)',
             'ROE(당기순이익)(%)',             'ROA(당기순이익)(%)',
                   '총자산(천원)',                  '유동자산(천원)',
                  '유형자산(천원)',                  '무형자산(천원)',
                   '총부채(천원)',                    '사채(천원)',
               '*단기차입부채(천원)',               '*장기차입부채(천원)',
                   '총자본(천원)',                 '자본잉여금(천원)',
                 '이익잉여금(천원)',              '현금및현금성자산(천원)',
          '영업활동으로인한현금흐름(천원)',          '투자활동으로인한현금흐름(천원)',
          '재무활동으로인한현금흐름(천원)',                  '현금흐름(천원)',
               '액면가(원,US센트)',                  '상장주식수(주)',
           '기말발행주식수 (보통)(주)', '시가총액 (티커-상장예정주식수 포함)(백만원)',
            '외국인보유비중(티커)(%)']
Length: 29, dtype: str


In [188]:
# Filter for required Item Names
required_items = ['영업이익(천원)', '매출액(천원)', 'ROE(당기순이익)(%)', '총부채(천원)', '총자본(천원)']

# Use the correct column name (with or without trailing space)
item_col = 'Item Name ' if 'Item Name ' in df_financial.columns else 'Item Name'
df_subset = df_financial[df_financial[item_col].isin(required_items)].copy()

print(f"Filtered to {len(df_subset)} rows with required Item Names")
print(f"\nItem Name counts:")
print(df_subset[item_col].value_counts())

Filtered to 19490 rows with required Item Names

Item Name counts:
Item Name 
매출액(천원)          3898
영업이익(천원)         3898
ROE(당기순이익)(%)    3898
총부채(천원)          3898
총자본(천원)          3898
Name: count, dtype: int64


In [189]:
# Pivot to get one row per Symbol with metrics as columns
# Use the correct column name (with or without trailing space)
item_col = 'Item Name ' if 'Item Name ' in df_financial.columns else 'Item Name'

df_pivot = df_subset.pivot_table(
    index=['Symbol', 'Symbol Name'],
    columns=item_col,
    values=latest_quarter,
    aggfunc='first'
).reset_index()

print(f"Pivoted shape: {df_pivot.shape}")
print(f"\nColumns after pivot:")
print(df_pivot.columns.tolist())
df_pivot.head()

Pivoted shape: (2156, 7)

Columns after pivot:
['Symbol', 'Symbol Name', 'ROE(당기순이익)(%)', '매출액(천원)', '영업이익(천원)', '총부채(천원)', '총자본(천원)']


Item Name,Symbol,Symbol Name,ROE(당기순이익)(%),매출액(천원),영업이익(천원),총부채(천원),총자본(천원)
0,A000010,신한은행,11.51,"5,118,324,000","1,466,075,000","556,153,695,000","38,363,620,000"
1,A000020,동화약품,0.24,"121,998,971","1,369,909","252,514,979","402,113,471"
2,A000030,우리은행,10.18,"4,367,485,000","953,514,000","460,766,828,000","29,313,198,000"
3,A000040,KR모터스,-29.63,"3,408,911","-1,261,648","55,147,568","41,008,288"
4,A000050,경방,8.20,"96,907,987","10,732,199","442,382,880","784,145,426"


In [190]:
# Rename columns for easier access
df_pivot.columns.name = None  # Remove the 'Item Name' label
column_mapping = {
    '영업이익(천원)': '영업이익',
    '매출액(천원)': '매출액',
    'ROE(당기순이익)(%)': 'ROE',
    '총부채(천원)': '총부채',
    '총자본(천원)': '총자본'
}

df_pivot = df_pivot.rename(columns=column_mapping)
print("Renamed columns:")
print(df_pivot.columns.tolist())

Renamed columns:
['Symbol', 'Symbol Name', 'ROE', '매출액', '영업이익', '총부채', '총자본']


In [191]:
# Ensure all tickers have 'A' prefix
df_pivot['Symbol'] = df_pivot['Symbol'].apply(lambda x: x if str(x).startswith('A') else f'A{x}')
print(f"Ensured all {len(df_pivot)} tickers have 'A' prefix")

# Calculate 영업이익률 and 부채비율
# Handle division by zero and missing values

# CRITICAL: Remove commas from numeric strings before converting
# Numbers in CSV are formatted like "8,488,799,000" (strings with commas)
for col in ['영업이익', '매출액', 'ROE', '총부채', '총자본']:
    # Remove commas from strings
    df_pivot[col] = df_pivot[col].apply(lambda x: str(x).replace(',', ''))
    # Convert to numeric, replacing errors with NaN
    df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')

print("Cleaned and converted numeric columns (removed commas)")

# Calculate 영업이익률 = (영업이익 / 매출액) × 100
df_pivot['영업이익률'] = (df_pivot['영업이익'] / df_pivot['매출액']) * 100

# Calculate 부채비율 = (총부채 / 총자본) × 100
df_pivot['부채비율'] = (df_pivot['총부채'] / df_pivot['총자본']) * 100

print("Calculated metrics:")
print(f"Total stocks: {len(df_pivot)}")
print(f"\nStocks with valid 영업이익률: {df_pivot['영업이익률'].notna().sum()}")
print(f"Stocks with valid ROE: {df_pivot['ROE'].notna().sum()}")
print(f"Stocks with valid 부채비율: {df_pivot['부채비율'].notna().sum()}")

# Display sample of calculated metrics
print("\nSample of calculated metrics:")
df_pivot[['Symbol', 'Symbol Name', '영업이익률', 'ROE', '부채비율']].head(10)

Ensured all 2156 tickers have 'A' prefix
Cleaned and converted numeric columns (removed commas)
Calculated metrics:
Total stocks: 2156

Stocks with valid 영업이익률: 2156
Stocks with valid ROE: 2117
Stocks with valid 부채비율: 2156

Sample of calculated metrics:


,Symbol,Symbol Name,영업이익률,ROE,부채비율
0,A000010,신한은행,28.643654,11.51,1449.690345
1,A000020,동화약품,1.122886,0.24,62.796946
2,A000030,우리은행,21.832107,10.18,1571.874990
3,A000040,KR모터스,-37.010294,-29.63,134.479079
4,A000050,경방,11.074628,8.20,56.415923
5,A000060,메리츠화재,26.926165,32.78,697.591536
6,A000070,삼양홀딩스,5.219980,4.28,72.662588
7,A000080,하이트진로,8.124152,11.46,186.296645
8,A000100,유한양행,3.861982,3.82,37.722812
9,A000110,제일은행,19.542443,6.82,1581.147260


## Step 3: Apply Filtering Criteria

In [192]:
# Drop rows with missing values in any of the three metrics
df_clean = df_pivot.dropna(subset=['영업이익률', 'ROE', '부채비율']).copy()

print(f"Stocks with complete data: {len(df_clean)} (dropped {len(df_pivot) - len(df_clean)} stocks with missing values)")

Stocks with complete data: 2117 (dropped 39 stocks with missing values)


In [193]:
# Apply individual filters and show statistics
print("Applying filters:\n")

opm_pass = df_clean['영업이익률'] >= OPM_MIN
print(f"영업이익률 >= {OPM_MIN}%: {opm_pass.sum():,} stocks pass")

roe_pass = df_clean['ROE'] >= ROE_MIN
print(f"ROE >= {ROE_MIN}%: {roe_pass.sum():,} stocks pass")

debt_pass = df_clean['부채비율'] <= DEBT_MAX
print(f"부채비율 <= {DEBT_MAX}%: {debt_pass.sum():,} stocks pass")

# Combine all filters
combined_mask = opm_pass & roe_pass & debt_pass
df_filtered = df_clean[combined_mask].copy()

print(f"\n{'='*50}")
print(f"Combined (all conditions): {len(df_filtered):,} stocks selected")
print(f"{'='*50}")

Applying filters:

영업이익률 >= 8.0%: 617 stocks pass
ROE >= 10.0%: 654 stocks pass
부채비율 <= 150.0%: 1,616 stocks pass

Combined (all conditions): 346 stocks selected


## Step 4: Extract and Display Selected Tickers

In [194]:
# Extract ticker list
selected_tickers = df_filtered['Symbol'].tolist()

print(f"\n선택된 종목 수: {len(selected_tickers)}")
print(f"\n선택된 종목 코드:")
print(selected_tickers)


선택된 종목 수: 346

선택된 종목 코드:
['A000220', 'A000240', 'A000300', 'A000640', 'A000660', 'A000700', 'A000990', 'A001060', 'A002460', 'A002620', 'A002800', 'A002840', 'A003090', 'A003100', 'A003160', 'A003230', 'A003720', 'A003780', 'A003850', 'A004800', 'A005180', 'A005430', 'A005710', 'A005750', 'A005930', 'A006730', 'A007340', 'A007540', 'A007660', 'A007700', 'A008930', 'A009540', 'A009780', 'A009900', 'A009970', 'A010120', 'A010240', 'A010400', 'A010620', 'A012030', 'A012510', 'A012750', 'A013030', 'A014580', 'A014680', 'A014940', 'A015230', 'A017960', 'A018290', 'A019010', 'A021240', 'A022100', 'A023160', 'A024950', 'A025540', 'A025560', 'A025880', 'A028050', 'A030000', 'A030190', 'A031510', 'A031980', 'A033500', 'A033780', 'A034950', 'A035150', 'A035250', 'A035420', 'A035460', 'A035610', 'A035890', 'A035900', 'A036120', 'A036170', 'A036530', 'A036640', 'A036890', 'A037270', 'A037350', 'A038010', 'A039030', 'A039420', 'A039560', 'A039840', 'A040420', 'A041510', 'A041830', 'A042000', 'A04

In [195]:
# Display summary table with calculated metrics
print("\n선택된 종목 상세 정보 (Top 20 by ROE):")
summary = df_filtered[['Symbol', 'Symbol Name', '영업이익률', 'ROE', '부채비율']].sort_values('ROE', ascending=False)
summary.head(20)


선택된 종목 상세 정보 (Top 20 by ROE):


,Symbol,Symbol Name,영업이익률,ROE,부채비율
997,A068790,DMS,16.091298,188.12,28.524174
2094,A458870,씨어스테크놀로지,42.742567,101.24,33.884989
957,A065560,녹원씨엔아이,26.284990,82.19,17.803386
1819,A300080,플리토,24.019090,79.50,65.519865
1766,A278470,에이피알,24.907234,78.89,76.975125
1350,A123330,제닉,23.976220,65.39,43.730999
779,A046940,우원개발,28.386469,64.88,86.318238
1627,A223250,드림씨아이에스,16.878971,63.77,64.681555
1498,A184230,SGA솔루션즈,27.833695,60.58,100.365574
1711,A257720,실리콘투,21.068246,60.21,59.636287


In [196]:
# Display summary statistics for selected stocks
print("\n선택된 종목의 지표 통계:")
summary_stats = df_filtered[['영업이익률', 'ROE', '부채비율']].describe()
summary_stats


선택된 종목의 지표 통계:


,영업이익률,ROE,부채비율
count,346.000000,346.000000,346.000000
mean,18.565590,20.876734,52.497833
std,10.381380,15.427631,35.394121
min,8.012476,10.000000,2.236574
25%,11.757813,12.655000,24.454339
50%,15.845573,16.200000,43.368920
75%,22.049127,23.625000,76.922444
max,86.042462,188.120000,148.784927


## Step 5: Load and Filter 퀀트데이터_260102.csv

In [197]:
# Load quant data
print("Loading 퀀트데이터_260102.csv...")
df_quant = pd.read_csv('data/퀀트데이터_260102.csv', encoding='utf-8-sig')

# Ensure all tickers have 'A' prefix
df_quant['코드'] = df_quant['코드'].apply(lambda x: x if str(x).startswith('A') else f'A{x}')
print(f"Ensured all {len(df_quant)} tickers have 'A' prefix")

print(f"Shape: {df_quant.shape}")
print(f"Columns: {len(df_quant.columns)} total")
print(f"\nFirst few columns: {list(df_quant.columns[:5])}")

Loading 퀀트데이터_260102.csv...
Ensured all 2700 tickers have 'A' prefix
Shape: (2700, 449)
Columns: 449 total

First few columns: ['코드', '회사명', '시장구분', '업종대', '업종소']


In [198]:
# Filter quant data by selected tickers
# Note: 퀀트데이터 uses '코드' column which should match 'Symbol' from 재무데이터
filtered_quant = df_quant[df_quant['코드'].isin(selected_tickers)].copy()

print(f"\n필터링된 퀀트데이터 행 수: {len(filtered_quant)}")
print(f"\nVerification: {len(filtered_quant)} rows (should match {len(selected_tickers)} selected tickers)")

# Check if all tickers were found
matched_tickers = set(filtered_quant['코드'].unique())
missing_tickers = set(selected_tickers) - matched_tickers

if missing_tickers:
    print(f"\nWarning: {len(missing_tickers)} tickers not found in 퀀트데이터:")
    print(list(missing_tickers)[:10])  # Show first 10
else:
    print("\nAll selected tickers found in 퀀트데이터! ✓")


필터링된 퀀트데이터 행 수: 343

Verification: 343 rows (should match 346 selected tickers)

['A194510', 'A065560', 'A102210']


In [199]:
# Optional: Filter for KOSPI 200 stocks only
# KOSPI 200 = top 200 KOSPI stocks by market cap (threshold: ~12,650억 won)
print("\n" + "="*60)
print("KOSPI 200 FILTERING")
print("="*60)

kospi_200 = filtered_quant[
    (filtered_quant['시장구분'] == '코스피') &
    (filtered_quant['시가총액(억)'] >= 12650)
].copy()

print(f"\nTotal filtered stocks: {len(filtered_quant)}")
print(f"KOSPI stocks: {len(filtered_quant[filtered_quant['시장구분'] == '코스피'])}")
print(f"KOSDAQ stocks: {len(filtered_quant[filtered_quant['시장구분'] == '코스닥'])}")
print(f"\nKOSPI 200 stocks (market cap >= 12,650억): {len(kospi_200)}")

if len(kospi_200) > 0:
    print(f"\nTop 10 KOSPI 200 stocks by market cap:")
    top_10 = kospi_200.nlargest(10, '시가총액(억)')[['코드', '회사명', '시가총액(억)']]
    print(top_10.to_string(index=False))
    
print("\n" + "="*60)


KOSPI 200 FILTERING

Total filtered stocks: 343
KOSPI stocks: 104
KOSDAQ stocks: 239

KOSPI 200 stocks (market cap >= 12,650억): 46

Top 10 KOSPI 200 stocks by market cap:
     코드         회사명  시가총액(억)
A005930        삼성전자  7606735
A000660      SK하이닉스  4928576
A207940    삼성바이오로직스   779077
A402340       SK스퀘어   517781
A035420       NAVER   387426
A267260    HD현대일렉트릭   295226
A009540    HD한국조선해양   278492
A064350        현대로템   211081
A033780        KT&G   165639
A010120 LS ELECTRIC   147750



In [200]:
# Display KOSPI 200 ticker codes
if len(kospi_200) > 0:
    kospi_200_tickers = kospi_200['코드'].tolist()
    print(f"\n코스피 200 종목 코드 ({len(kospi_200_tickers)}개):")
    print(kospi_200_tickers)
    
    # Show detailed list with company names
    print(f"\n코스피 200 종목 상세 리스트:")
    kospi_200_list = kospi_200[['코드', '회사명', '시가총액(억)']].sort_values('시가총액(억)', ascending=False)
    print(kospi_200_list.to_string(index=False))
else:
    print("\n코스피 200 종목이 없습니다.")


코스피 200 종목 코드 (46개):
['A005930', 'A000660', 'A207940', 'A402340', 'A035420', 'A267260', 'A009540', 'A064350', 'A033780', 'A010120', 'A042700', 'A259960', 'A326030', 'A003230', 'A007660', 'A278470', 'A443060', 'A161390', 'A021240', 'A128940', 'A028050', 'A022100', 'A062040', 'A271560', 'A035250', 'A111770', 'A000990', 'A071970', 'A012510', 'A014680', 'A012750', 'A439260', 'A383220', 'A008930', 'A009970', 'A081660', 'A030000', 'A000240', 'A353200', 'A051600', 'A069620', 'A483650', 'A004800', 'A161890', 'A017960', 'A007340']

코스피 200 종목 상세 리스트:
     코드         회사명  시가총액(억)
A005930        삼성전자  7606735
A000660      SK하이닉스  4928576
A207940    삼성바이오로직스   779077
A402340       SK스퀘어   517781
A035420       NAVER   387426
A267260    HD현대일렉트릭   295226
A009540    HD한국조선해양   278492
A064350        현대로템   211081
A033780        KT&G   165639
A010120 LS ELECTRIC   147750
A042700       한미반도체   137726
A259960        크래프톤   117562
A326030      SK바이오팜    96247
A003230        삼양식품    96121
A007660      이수페

## Step 5.6: Apply Valuation Filters (Final Selection)

In [201]:
# Apply valuation filters: PER <= 10 AND PBR <= 2.0
# This reduces from ~46 KOSPI 200 stocks to ~10 deep value stocks
print("\n" + "="*60)
print("VALUATION FILTERING (FINAL SELECTION)")
print("="*60)

# Define valuation thresholds
PER_MAX = 10.0    # PER <= 10 (deeply undervalued)
PBR_MAX = 2.0     # PBR <= 2.0 (not too expensive vs book value)

print(f"\nApplying valuation filters:")
print(f"  PER <= {PER_MAX}")
print(f"  PBR <= {PBR_MAX}")

# Apply filters
final_selection = kospi_200[
    (kospi_200['PER'] <= PER_MAX) &
    (kospi_200['PBR'] <= PBR_MAX)
].copy()

print(f"\nKOSPI 200 stocks: {len(kospi_200)}")
print(f"Final value selection: {len(final_selection)}")
print(f"Reduction: {len(kospi_200) - len(final_selection)} stocks filtered out")
print("\n" + "="*60)


VALUATION FILTERING (FINAL SELECTION)

Applying valuation filters:
  PER <= 10.0
  PBR <= 2.0

KOSPI 200 stocks: 46
Final value selection: 10
Reduction: 36 stocks filtered out



In [202]:
# Display final selection with key metrics
print("\n최종 선택 종목 (가치주):")
print("="*80)

if len(final_selection) > 0:
    # Show stocks sorted by PER (cheapest first)
    display_cols = ['코드', '회사명', 'PER', 'PBR', 'OPM (%)', 'ROE (%)', '부채 비율 (%)', '시가총액(억)']
    final_display = final_selection[display_cols].sort_values('PER')
    
    print(final_display.to_string(index=False))
    
    # Display ticker list
    final_tickers = final_selection['코드'].tolist()
    print(f"\n\n최종 선택 종목 코드 ({len(final_tickers)}개):")
    print(final_tickers)
    
    # Summary statistics
    print("\n\n최종 선택 종목 평균 지표:")
    print(f"  평균 PER: {final_selection['PER'].mean():.2f}x")
    print(f"  평균 PBR: {final_selection['PBR'].mean():.2f}x")
    print(f"  평균 OPM: {final_selection['OPM (%)'].mean():.2f}%")
    print(f"  평균 ROE: {final_selection['ROE (%)'].mean():.2f}%")
    print(f"  평균 부채비율: {final_selection['부채 비율 (%)'].mean():.2f}%")
else:
    print("조건을 만족하는 종목이 없습니다.")
    
print("="*80)


최종 선택 종목 (가치주):
     코드         회사명  PER  PBR  OPM (%)  ROE (%)  부채 비율 (%)  시가총액(억)
A007340     DN오토모티브 5.56 0.73    13.75    13.19     119.87    13809
A004800          효성 5.93 0.82    17.44    13.78      83.62    18833
A383220         F&F 7.24 1.46    23.94    20.23      42.85    25398
A161390 한국타이어앤테크놀로지 7.42 0.62     9.67     8.31     102.17    70609
A009970     영원무역홀딩스 7.64 0.89    12.20    11.62      31.89    24785
A271560         오리온 7.69 1.12    16.83    14.61      16.90    40762
A000240      한국앤컴퍼니 7.71 0.52    26.05     6.74       9.86    23829
A111770        영원무역 7.87 0.93     9.81    11.79      39.03    35139
A028050       삼성E&A 8.30 1.00     9.16    12.06     101.73    45570
A259960        크래프톤 9.43 1.66    41.91    17.64      15.96   117562


최종 선택 종목 코드 (10개):
['A259960', 'A161390', 'A028050', 'A271560', 'A111770', 'A383220', 'A009970', 'A000240', 'A004800', 'A007340']


최종 선택 종목 평균 지표:
  평균 PER: 7.48x
  평균 PBR: 0.98x
  평균 OPM: 18.08%
  평균 ROE: 13.00%
  평균 부채비율: 56.39%


## Step 6: Display Final Results

In [203]:
# Display first 20 rows of filtered quant data
print("필터링된 퀀트데이터 (처음 20개):")
filtered_quant.head(20)

필터링된 퀀트데이터 (처음 20개):


,코드,회사명,시장구분,업종대,업종소,주가(원),시가총액(억),시가총액(억) 우선주포함,주식수 (단위:만주),유통주식 비중 (%),...,EPS 24년1Q,EPS 24년2Q,EPS 24년3Q,EPS 24년4Q,EPS 25년1Q,EPS 25년2Q,EPS 25년3Q(E),25년3Q 매출액,25년3Q 영업이익,25년3Q 지배순이익
0,A005930,삼성전자,코스피,반도체 관련장비 및 부품,종합 반도체,128500,7606735,8377015,591964,75.7,...,1109.09,1615.24,1638.51,1269.08,1356.23,833.50,2028.24,860617.47,121660.62,120064.61
1,A000660,SK하이닉스,코스피,반도체 관련장비 및 부품,종합 반도체,677000,4928576,4928576,72800,74.1,...,2636.33,5659.71,7896.51,10989.63,11136.06,9611.55,17301.04,244489.29,113833.90,125951.95
3,A207940,삼성바이오로직스,코스피,제약 및 바이오,바이오,1683000,779077,779077,4629,25.6,...,3708.86,6575.48,5468.71,6647.88,7765.74,6707.69,12409.76,16602.21,7288.48,5744.60
6,A402340,SK스퀘어,코스피,통신,통신서비스,392000,517781,517781,13209,67.7,...,2481.64,5496.41,8434.18,11071.80,12100.80,10866.04,18815.75,30746.54,26455.08,24853.18
13,A035420,NAVER,코스피,IT서비스,인터넷서비스,247000,387426,387426,15685,85.9,...,3143.70,2083.64,3236.44,3496.04,2681.46,3085.02,4630.94,31380.55,5706.47,7263.74
19,A267260,HD현대일렉트릭,코스피,기계,기계,819000,295226,295226,3605,62.4,...,2607.31,4479.36,3241.62,3585.92,4274.19,3951.09,5301.02,9954.32,2470.76,1910.87
20,A009540,HD한국조선해양,코스피,조선,조선,393500,278492,278492,7077,63.2,...,2669.75,4127.71,2134.19,7633.05,6999.79,5036.36,8948.36,75814.36,10537.75,6333.03
23,A196170,알테오젠,코스닥,제약 및 바이오,바이오,457000,244521,244521,5351,79.6,...,395.50,-54.93,-174.89,1003.41,1559.99,-618.99,417.62,490.13,266.71,223.45
30,A064350,현대로템,코스피,기계,중장비 및 관련품,193400,211081,211081,10914,66.2,...,515.30,928.31,954.49,1329.98,1451.18,1746.55,1818.66,16196.35,2777.44,1984.93
39,A033780,KT&G,코스피,식음료,식료품,140400,165639,165639,11798,74.9,...,2197.21,2400.48,1851.12,2594.89,2094.14,1174.21,3553.70,18268.96,4652.89,4192.54


In [204]:
# Display final selection details
print("\n" + "="*60)
print("FINAL SELECTION - VALUE STOCKS")
print("="*60)
print(f"\n총 {len(final_selection)}개 종목 선택됨")
print("\n상세 정보:")
print("-"*60)

# Show all final stocks with key metrics
display_cols = ['코드', '회사명', 'PER', 'PBR', 'OPM (%)', 'ROE (%)', '부채 비율 (%)', '시가총액(억)']
final_display = final_selection[display_cols].sort_values('PER')
print(final_display.to_string(index=False))

print("\n" + "="*60)
print(f"평균 PER: {final_selection['PER'].mean():.2f}x | 평균 PBR: {final_selection['PBR'].mean():.2f}x")
print(f"평균 OPM: {final_selection['OPM (%)'].mean():.2f}% | 평균 ROE: {final_selection['ROE (%)'].mean():.2f}%")
print("="*60)


FINAL SELECTION - VALUE STOCKS

총 10개 종목 선택됨

상세 정보:
------------------------------------------------------------
     코드         회사명  PER  PBR  OPM (%)  ROE (%)  부채 비율 (%)  시가총액(억)
A007340     DN오토모티브 5.56 0.73    13.75    13.19     119.87    13809
A004800          효성 5.93 0.82    17.44    13.78      83.62    18833
A383220         F&F 7.24 1.46    23.94    20.23      42.85    25398
A161390 한국타이어앤테크놀로지 7.42 0.62     9.67     8.31     102.17    70609
A009970     영원무역홀딩스 7.64 0.89    12.20    11.62      31.89    24785
A271560         오리온 7.69 1.12    16.83    14.61      16.90    40762
A000240      한국앤컴퍼니 7.71 0.52    26.05     6.74       9.86    23829
A111770        영원무역 7.87 0.93     9.81    11.79      39.03    35139
A028050       삼성E&A 8.30 1.00     9.16    12.06     101.73    45570
A259960        크래프톤 9.43 1.66    41.91    17.64      15.96   117562

평균 PER: 7.48x | 평균 PBR: 0.98x
평균 OPM: 18.08% | 평균 ROE: 13.00%


In [205]:
# Final selected tickers
print("\n" + "="*60)
print("FINAL SELECTED TICKERS")
print("="*60)
print(f"\n최종 선택된 {len(final_selection)}개 종목:")
print(final_selection['코드'].tolist())
print("\n" + "="*60)


FINAL SELECTED TICKERS

최종 선택된 10개 종목:
['A259960', 'A161390', 'A028050', 'A271560', 'A111770', 'A383220', 'A009970', 'A000240', 'A004800', 'A007340']

